<a href="https://colab.research.google.com/github/vinaykrshnn-git2026/soccer_stats_norwestfc/blob/main/01_get_match_narrative.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
# This will trigger a pop-up window asking for permission to access your files
drive.mount('/content/drive')

# 2. Define the path to your file
# Adjust the path below to match the exact location in your My Drive
file_path = '/content/drive/MyDrive/Norwest FC/2026 Season/Player Time Tracker 2026.xlsx'

# 3. Read ONLY the 28-March sheet
# We use header=None first so the LLM can see the entire layout including metadata
try:
    df_28_march = pd.read_excel(file_path, sheet_name='28-March', header=None)

    # 4. Convert the grid to a single text block for the LLM
    raw_data_string = df_28_march.to_string()

    print("Successfully read 28-March. Data length:", len(raw_data_string))
    print("\n--- Preview of Raw Data for Agent ---")
    print(raw_data_string[:500]) # Previewing the first 500 characters

except Exception as e:
    print(f"Error: Could not find the sheet or file. Details: {e}")

In [ ]:
from openai import OpenAI
from google.colab import userdata

# Set up your client
client = openai.OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# The Raw Data from your previous cell
match_data = raw_data_string

# The Prompt: This is the "Agent's Instructions"
prompt = f"""
You are a Data Analyst for a youth soccer team. Convert the following raw spreadsheet data into a natural language match report.

INSTRUCTIONS:
1. Extract: Match Date, Opponent, and the Final Score (GF vs GA).
2. Key Events: Identify Goal Scorers and the 'Player of the Match' (and who awarded it).
3. Playing Time Analysis: Identify the standard playing time for each player(e.g., 30 mins). Explicitly name any players who played LESS than the standard time and explain why based on the substitution notes (e.g., 'Patrick -> Arrad').
4. GoalKeepers: Note who shared the goalkeeping duties.

RAW DATA BLOCK:
{match_data}
"""

response = client.chat.completions.create(
    model="gpt-4o", # GPT-4o is excellent at reading text-based tables
    messages=[
        {"role": "system", "content": "You are a helpful coaching assistant."},
        {"role": "user", "content": prompt}
    ]
)

# This is the "Narrative" we will eventually vectorize
narrative_28_march = response.choices[0].message.content
print("--- GENERATED NARRATIVE ---")
print(narrative_28_march)